# Two_Tower_BPR_Watchstatus_Profile

The core of this recommendation system is a **Two-Tower Neural Network**. This architecture is widely used for recommender systems due to its efficiency and effectiveness, especially for handling large catalogs of items.

### How it Works:

The Two-Tower model consists of two independent neural network 'towers':

1.  **User Tower**: Takes user-specific features as input and outputs a dense vector representation (embedding) for each user.
2.  **Item Tower**: Takes item-specific features as input and outputs a dense vector representation (embedding) for each item.

During training, the goal is to learn these embeddings such that the dot product (or cosine similarity) between a user's embedding and an item's embedding reflects the user's preference for that item. A higher dot product typically indicates a stronger preference.

### Model Components (`TwoTowerSimple`):

*   **Input Features**:
    *   **User Side**: User ID embedding (`user_id_emb`) and pre-computed dense user features (`user_dense_table`). The dense user features include basic statistics (rating count, mean, std), watching status distribution, and a positive-item preference vector derived from items the user liked in the training set.
    *   **Item Side**: Item ID embedding (`item_id_emb`) and pre-computed dense item features (`item_dense_table`). These are typically features like genres, tags, or other descriptive attributes of the anime.

*   **MLP Layers**:
    *   **`user_mlp`**: A multi-layer perceptron that takes the concatenated user ID embedding and dense user features, processes them through hidden layers (with ReLU activation and Dropout for regularization), and outputs the final user embedding (`EMB_DIM` dimensions).
    *   **`item_mlp`**: Similarly, an MLP that processes the concatenated item ID embedding and dense item features to output the final item embedding (`EMB_DIM` dimensions).

*   **Bias Terms**:
    *   `user_bias`: A learned bias for each user, accounting for their general tendency to rate items higher or lower.
    *   `item_bias`: A learned bias for each item, accounting for its general popularity or quality.
    *   `global_bias`: An overall constant bias.

### Scoring and Recommendation:

*   The model calculates a **score** for a given user-item pair by taking the dot product of their respective learned embeddings from the user and item towers, and then adding the user bias, item bias, and global bias.
    `score(u, i) = user_vector(u) ⋅ item_vector(i) + user_bias(u) + item_bias(i) + global_bias`

*   For **recommendation**, once the model is trained, the user tower can generate an embedding for any user, and the item tower can generate embeddings for all items in the catalog. To find recommendations for a user, the model computes the scores between the user's embedding and all item embeddings, ranks them, and returns the top-K items.

## Special Feature: Utilization of Watch Status

A key characteristic of this model is how it leverages a user's **Watch Status** as part of their user features, allowing for a more nuanced understanding of user preferences.

In the `user_dense_table`, in addition to basic rating statistics and positive item preference vectors, the model also incorporates each user's **distribution of watch statuses**. Specifically:

*   For each user, the model calculates the **proportion or average** of different watch statuses (e.g., 'Completed', 'Currently Watching', 'Dropped', 'On Hold', 'Plan to Watch') across all anime they have interacted with.
*   These proportions form a user's **watch status profile**. For instance, one user might have a high proportion of completed anime, while another might have many anime in their 'Plan to Watch' list, or a higher proportion of 'Dropped' titles.
*   This 'watch status profile' is used as an important **dense user feature** input to the **User Tower**.

By doing so, the model not only considers what a user likes (high ratings) but also their **viewing habits and tendencies**. For example:
*   A user who tends to 'complete' anime might prefer anime with complete and coherent storylines.
*   A user with a large number of 'Plan to Watch' anime might show higher interest in new or popular works.
*   A user with a high proportion of 'Dropped' anime might have lower tolerance for certain genres or lose interest in works that aren't engaging from the start.

These detailed viewing habit insights help the model more accurately capture a user's potential preferences, thereby providing recommendations that better align with their individual behavioral patterns.

In [ ]:
import os
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. Config
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# -------- Your actual file paths --------
TRAIN_PATH = Path('/content/drive/MyDrive/yuran_files/train_complete.csv')
VAL_PATH   = Path('/content/drive/MyDrive/yuran_files/val_complete.csv')
TEST_PATH  = Path('/content/drive/MyDrive/yuran_files/test_complete.csv')
MASTER_ANIME_PATH = Path('/content/drive/MyDrive/MASTER_ANIME_TOWER_FEATURES_1BASED.npy')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ----------------------------
# SETTINGS
# ----------------------------
POS_THRESHOLD = 6.0 # Lowered from 8.0 to include more positive interactions

BATCH_SIZE = 4096 # Increased batch size
EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-6
PATIENCE = 2

EMB_DIM = 64
HIDDEN_DIM = 256
DROPOUT = 0.10

ITEM_FEAT_DIM_TO_USE = 883
MAX_TRAIN_POS_PER_USER = None   # Removed cap to use all positive interactions
SAMPLE_DATA_FRAC = 1.0 # New: Use all of the data

# Full-catalog evaluation settings
EVAL_K = 10
EVAL_MAX_USERS = None # Evaluate on all users

# ============================================================
# 2. Load train / val / test
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

# Apply sampling if SAMPLE_DATA_FRAC is set
if SAMPLE_DATA_FRAC < 1.0:
    train_df = train_df.sample(frac=SAMPLE_DATA_FRAC, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=SAMPLE_DATA_FRAC, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(frac=SAMPLE_DATA_FRAC, random_state=SEED).reset_index(drop=True)

# Normalize rating column name if needed
if "score" in train_df.columns and "rating" not in train_df.columns:
    train_df = train_df.rename(columns={"score": "rating"})
if "score" in val_df.columns and "rating" not in val_df.columns:
    val_df = val_df.rename(columns={"score": "rating"})
if "score" in test_df.columns and "rating" not in test_df.columns:
    test_df = test_df.rename(columns={"score": "rating"})

# Required base columns
required_cols = ["user_idx", "anime_idx", "rating"]

# Optional watching-status columns
status_cols = [
    "watch_status_Completed",
    "watch_status_Currently Watching",
    "watch_status_Dropped",
    "watch_status_On Hold",
    "watch_status_Plan to Watch",
]

# Optional extra interaction columns
optional_cols = [
    "watched_episodes",
    "Episodes",
    "watched_episode_ratio",
    "rating_no_negative"
]

for c in required_cols:
    if c not in train_df.columns:
        raise ValueError(f"Missing column in train_complete.csv: {c}")
    if c not in val_df.columns:
        raise ValueError(f"Missing column in val_complete.csv: {c}")
    if c not in test_df.columns:
        raise ValueError(f"Missing column in test_complete.csv: {c}")

keep_cols = required_cols.copy()
for c in status_cols + optional_cols:
    if c in train_df.columns and c in val_df.columns and c in test_df.columns:
        keep_cols.append(c)

train_df = train_df[keep_cols].copy()
val_df = val_df[keep_cols].copy()
test_df = test_df[keep_cols].copy()

for df in [train_df, val_df, test_df]:
    df["user_idx"] = df["user_idx"].astype(int)
    df["anime_idx"] = df["anime_idx"].astype(int)
    df["rating"] = df["rating"].astype(float)

    for c in status_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype(float)

    for c in optional_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

# --- New: Remap user_idx and anime_idx to contiguous IDs after sampling ---
# Get all unique user and anime IDs from the sampled dataframes
all_unique_users = pd.concat([train_df['user_idx'], val_df['user_idx'], test_df['user_idx']]).unique()
all_unique_animes = pd.concat([train_df['anime_idx'], val_df['anime_idx'], test_df['anime_idx']]).unique()

# Create mappings from old ID to new contiguous ID (0-based)
user_id_map = {old_id: new_id for new_id, old_id in enumerate(sorted(all_unique_users))}
anime_id_map = {old_id: new_id for new_id, old_id in enumerate(sorted(all_unique_animes))}

# Apply mappings to dataframes
for df in [train_df, val_df, test_df]:
    df['user_idx'] = df['user_idx'].map(user_id_map)
    df['anime_idx'] = df['anime_idx'].map(anime_id_map)
    # Drop rows where mapping resulted in NaN (shouldn't happen if all_unique_* covers everything)
    df.dropna(subset=['user_idx', 'anime_idx'], inplace=True)
    df['user_idx'] = df['user_idx'].astype(int)
    df['anime_idx'] = df['anime_idx'].astype(int)

# Recalculate NUM_USERS and MAX_ANIME_IDX based on the new, contiguous IDs
NUM_USERS = len(user_id_map) # New NUM_USERS is simply the count of unique users
MAX_ANIME_IDX = len(anime_id_map) - 1 # New MAX_ANIME_IDX is max of remapped anime_idx
NUM_ITEMS = len(anime_id_map) # Define NUM_ITEMS here after remapping

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)
print("NUM_USERS (remapped):", NUM_USERS)
print("MAX_ANIME_IDX (remapped):", MAX_ANIME_IDX)
# --- End of New remapping logic ---


# ============================================================
# 3. Load MASTER_ANIME features
# ============================================================
original_master_anime_full = np.load(MASTER_ANIME_PATH).astype(np.float32)
print("Original master_anime shape (before remapping):", original_master_anime_full.shape)

# Create a new master_anime array that aligns with the remapped anime_idx
# Initialize with zeros, then fill in features for items that exist in the remapped data
remapped_master_anime = np.zeros((NUM_ITEMS, original_master_anime_full.shape[1]), dtype=np.float32)

# Fill remapped_master_anime with features from original_master_anime_full based on the anime_id_map
# Note: original_master_anime_full is 1-based, remapped_master_anime will be 0-based
for old_id, new_id in anime_id_map.items():
    if old_id < original_master_anime_full.shape[0]: # Check if old_id is within bounds of original features
        remapped_master_anime[new_id] = original_master_anime_full[old_id]
    # If old_id is out of bounds, the corresponding row in remapped_master_anime remains zeros

master_anime = remapped_master_anime[:, :ITEM_FEAT_DIM_TO_USE]

# NUM_ITEMS = master_anime.shape[0] # This line is no longer needed here
ITEM_FEAT_DIM = master_anime.shape[1]

master_anime = np.nan_to_num(master_anime, nan=0.0, posinf=0.0, neginf=0.0)

if NUM_ITEMS > 1:
    # Only normalize if there are actual items (excluding the potential 0-indexed placeholder)
    # If new_id 0 corresponds to a real item, then we normalize all. If it's a placeholder, skip.
    # Given we map all unique items to 0..N-1, all these items are 'real'.
    # We don't have a specific 'dummy' 0 item anymore in the remapped master_anime.
    item_mu = master_anime.mean(axis=0, keepdims=True)
    item_sigma = master_anime.std(axis=0, keepdims=True) + 1e-6
    master_anime = (master_anime - item_mu) / item_sigma

print("Aligned master_anime shape (remapped and selected features):", master_anime.shape)

# ============================================================
# 4. Build SIMPLE user features from TRAIN ONLY
#    + add watching-status profile
# ============================================================
# train_pos_df now contains remapped indices
train_pos_df = train_df[train_df["rating"] >= POS_THRESHOLD].copy()
if len(train_pos_df) == 0:
    raise ValueError(f"No positive samples found in train_complete.csv with POS_THRESHOLD={POS_THRESHOLD}")

if MAX_TRAIN_POS_PER_USER is not None:
    train_pos_df = (
        train_pos_df.groupby("user_idx", group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), MAX_TRAIN_POS_PER_USER), random_state=SEED), include_groups=False)
        .reset_index(drop=True)
    )

# Basic user stats
# user_stats_df now uses remapped user_idx
user_stats_df = train_df.groupby("user_idx").agg(
    user_rating_count=("rating", "count"),
    user_rating_mean=("rating", "mean"),
    user_rating_std=("rating", "std")
).fillna(0.0)

# Optional watched_episode_ratio stats
extra_user_stats = []
if "watched_episode_ratio" in train_df.columns:
    extra_user_stats.append(
        train_df.groupby("user_idx").agg(
            user_watch_ratio_mean=("watched_episode_ratio", "mean"),
            user_watch_ratio_std=("watched_episode_ratio", "std")
        ).fillna(0.0)
    )

# Watching-status distribution per user
status_profile_df = None
available_status_cols = [c for c in status_cols if c in train_df.columns]
if len(available_status_cols) > 0:
    # average of one-hot statuses = proportion of each status
    status_profile_df = train_df.groupby("user_idx")[available_status_cols].mean().fillna(0.0)

# Positive-item preference vector (uses remapped indices and remapped master_anime)
item_pref_sum = np.zeros((NUM_USERS, ITEM_FEAT_DIM), dtype=np.float32)
item_pref_weight = np.zeros(NUM_USERS, dtype=np.float32)

for row in train_pos_df.itertuples(index=False):
    u = int(row.user_idx)
    i = int(row.anime_idx)
    r = float(row.rating)

    if i < 0 or i >= NUM_ITEMS: # Changed condition because indices are now 0-based and contiguous
        continue

    # stronger weight for higher rating
    w = max((r - POS_THRESHOLD + 1.0), 1.0)
    item_pref_sum[u] += w * master_anime[i]
    item_pref_weight[u] += w

user_pref_vec = np.zeros((NUM_USERS, ITEM_FEAT_DIM), dtype=np.float32)
mask = item_pref_weight > 0
user_pref_vec[mask] = item_pref_sum[mask] / item_pref_weight[mask][:, None]

# Assemble user feature blocks
user_basic = np.zeros((NUM_USERS, user_stats_df.shape[1]), dtype=np.float32)
user_basic[user_stats_df.index.values] = user_stats_df.values.astype(np.float32)

feature_blocks = [user_basic]

for extra_df in extra_user_stats:
    arr = np.zeros((NUM_USERS, extra_df.shape[1]), dtype=np.float32)
    arr[extra_df.index.values] = extra_df.values.astype(np.float32)
    feature_blocks.append(arr)

if status_profile_df is not None:
    status_arr = np.zeros((NUM_USERS, status_profile_df.shape[1]), dtype=np.float32)
    status_arr[status_profile_df.index.values] = status_profile_df.values.astype(np.float32)
    feature_blocks.append(status_arr)

feature_blocks.append(user_pref_vec)

user_dense = np.concatenate(feature_blocks, axis=1)
user_dense = np.nan_to_num(user_dense, nan=0.0, posinf=0.0, neginf=0.0)

nz = (np.abs(user_dense).sum(axis=1) > 0)
if nz.any():
    mu = user_dense[nz].mean(axis=0, keepdims=True)
    sigma = user_dense[nz].std(axis=0, keepdims=True) + 1e-6
    user_dense[nz] = (user_dense[nz] - mu) / sigma

USER_FEAT_DIM = user_dense.shape[1]

print("user_dense shape:", user_dense.shape)
print("train_pos_df shape after cap:", train_pos_df.shape)
print("Included status columns:", available_status_cols)

# ============================================================
# 5. Prepare positives / seen maps
# ============================================================
# These dataframes now contain remapped indices
user_seen_train = train_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

val_pos_df = val_df[val_df["rating"] >= POS_THRESHOLD].copy()
val_pos_by_user = val_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

test_pos_df = test_df[test_df["rating"] >= POS_THRESHOLD].copy()
test_pos_by_user = test_pos_df.groupby("user_idx")["anime_idx"].apply(set).to_dict()

print("Train positives:", train_pos_df.shape)
print("Val positives:", val_pos_df.shape)
print("Test positives:", test_pos_df.shape)

# ============================================================
# 6. BPR dataset
# ============================================================
class TwoTowerBPRDataset(Dataset):
    def __init__(self, pos_df, user_seen_map, num_items):
        self.users = pos_df["user_idx"].to_numpy(np.int64)
        self.pos_items = pos_df["anime_idx"].to_numpy(np.int64)
        self.user_seen = user_seen_map
        self.num_items = num_items

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx])
        p = int(self.pos_items[idx])

        seen = self.user_seen.get(u, set())
        # Negative item sampling should now use NUM_ITEMS which is based on remapped contiguous IDs
        n = np.random.randint(0, self.num_items) # Changed from 1 to 0 to align with new 0-based contiguous indexing
        while n in seen:
            n = np.random.randint(0, self.num_items)

        return (
            torch.tensor(u, dtype=torch.long),
            torch.tensor(p, dtype=torch.long),
            torch.tensor(n, dtype=torch.long)
        )

train_loader = DataLoader(
    TwoTowerBPRDataset(train_pos_df, user_seen_train, NUM_ITEMS),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

# ============================================================
# 7. Evaluation helpers (FULL CATALOG)
# ============================================================
def _dcg_at_k(binary_hits):
    if len(binary_hits) == 0:
        return 0.0
    denom = np.log2(np.arange(2, len(binary_hits) + 2))
    return float((binary_hits / denom).sum())

@torch.no_grad()
def precompute_all_item_scores_matrix(model, item_batch_size=2048):
    model.eval()

    # item_ids_np should now cover all remapped item IDs from 0 to NUM_ITEMS-1
    item_ids_np = np.arange(0, NUM_ITEMS, dtype=np.int64) # Changed from 1 to 0
    item_vecs = []

    for start in range(0, len(item_ids_np), item_batch_size):
        batch_ids_np = item_ids_np[start:start + item_batch_size]
        batch_ids = torch.tensor(batch_ids_np, dtype=torch.long, device=DEVICE)
        batch_vecs = model.item_vector(batch_ids)
        item_vecs.append(batch_vecs)

    item_vecs = torch.cat(item_vecs, dim=0)
    return item_ids_np, item_vecs

@torch.no_grad()
def evaluate_full_catalog_at_k(model, pos_by_user, seen_items_by_user, k=10, max_users=None):
    if len(pos_by_user) == 0:
        return 0.0, 0.0, 0.0, 0.0

    model.eval()

    users = list(pos_by_user.keys())
    if max_users is not None and len(users) > max_users:
        users = list(np.random.choice(users, size=max_users, replace=False))

    item_ids_np, all_item_vecs = precompute_all_item_scores_matrix(model)

    precisions, recalls, hitrates, ndcgs = [], [], [], []
    user_batch_size = 256

    for start in range(0, len(users), user_batch_size):
        batch_users = users[start:start + user_batch_size]
        u_tensor = torch.tensor(batch_users, dtype=torch.long, device=DEVICE)

        user_vecs = model.user_vector(u_tensor)
        scores = torch.matmul(user_vecs, all_item_vecs.T).cpu().numpy()

        for row_idx, u in enumerate(batch_users):
            u = int(u)
            pos_items = np.array(list(pos_by_user.get(u, set())), dtype=np.int64)
            if len(pos_items) == 0:
                continue

            seen = seen_items_by_user.get(u, set())
            row_scores = scores[row_idx].copy()

            if len(seen) > 0:
                # seen_mask should check against item_ids_np (0 to NUM_ITEMS-1)
                seen_mask = np.isin(item_ids_np, list(seen))
                row_scores[seen_mask] = -1e12

            top_idx = np.argpartition(row_scores, -k)[-k:]
            top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
            top_items = item_ids_np[top_idx]

            hits = np.isin(top_items, pos_items).astype(np.float32)
            num_hits = float(hits.sum())

            precisions.append(num_hits / k)
            recalls.append(num_hits / max(len(pos_items), 1))
            hitrates.append(1.0 if num_hits > 0 else 0.0)

            dcg = _dcg_at_k(hits)
            ideal_hits = np.ones(min(k, len(pos_items)), dtype=np.float32)
            idcg = _dcg_at_k(ideal_hits)
            ndcgs.append((dcg / idcg) if idcg > 0 else 0.0)

    if len(precisions) == 0:
        return 0.0, 0.0, 0.0, 0.0

    return (
        float(np.mean(precisions)),
        float(np.mean(recalls)),
        float(np.mean(hitrates)),
        float(np.mean(ndcgs))
    )

# ============================================================
# 8. Two-tower model
# ============================================================
class TwoTowerSimple(nn.Module):
    def __init__(self, num_users, num_items, user_dense_np, item_dense_np):
        super().__init__()

        self.user_id_emb = nn.Embedding(num_users, EMB_DIM)
        self.item_id_emb = nn.Embedding(num_items, EMB_DIM)

        self.register_buffer("user_dense_table", torch.tensor(user_dense_np, dtype=torch.float32))
        self.register_buffer("item_dense_table", torch.tensor(item_dense_np, dtype=torch.float32))

        user_in_dim = EMB_DIM + self.user_dense_table.shape[1]
        item_in_dim = EMB_DIM + self.item_dense_table.shape[1]

        self.user_mlp = nn.Sequential(
            nn.Linear(user_in_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )

        self.item_mlp = nn.Sequential(
            nn.Linear(item_in_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, EMB_DIM)
        )

        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_id_emb.weight, std=0.02)
        nn.init.normal_(self.item_id_emb.weight, std=0.02)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def user_vector(self, u_idx):
        return self.user_mlp(torch.cat([self.user_id_emb(u_idx), self.user_dense_table[u_idx]], dim=1))

    def item_vector(self, i_idx):
        return self.item_mlp(torch.cat([self.item_id_emb(i_idx), self.item_dense_table[i_idx]], dim=1))

    def score(self, u_idx, i_idx):
        u_vec = self.user_vector(u_idx)
        i_vec = self.item_vector(i_idx)
        dot = (u_vec * i_vec).sum(dim=1);
        return dot + self.user_bias(u_idx).squeeze(-1) + self.item_bias(i_idx).squeeze(-1) + self.global_bias

# ============================================================
# 9. Train
# ============================================================
model = TwoTowerSimple(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    user_dense_np=user_dense,
    item_dense_np=master_anime
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_metric = -np.inf
best_state = None
best_epoch = -1
patience_counter = 0

print(f"Start training | users={NUM_USERS} | items={NUM_ITEMS} | POS_THRESHOLD={POS_THRESHOLD}")

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    for u, p, n in train_loader:
        u = u.to(DEVICE, non_blocking=True)
        p = p.to(DEVICE, non_blocking=True)
        n = n.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        pos_score = model.score(u, p)
        neg_score = model.score(u, n)

        loss = -F.logsigmoid(pos_score - neg_score).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        losses.append(loss.item())

    val_p, val_r, val_hr, val_ndcg = evaluate_full_catalog_at_k(
        model=model,
        pos_by_user=val_pos_by_user,
        seen_items_by_user=user_seen_train,
        k=EVAL_K,
        max_users=EVAL_MAX_USERS
    )

    print(
        f"Epoch {epoch:02d} | "
        f"bpr_loss={np.mean(losses):.4f} | "
        f"Val P@{EVAL_K}={val_p:.4f} | "
        f"Val R@{EVAL_K}={val_r:.4f} | "
        f"Val HR@{EVAL_K}={val_hr:.4f} | "
        f"Val NDCG@{EVAL_K}={val_ndcg:.4f}"
    )

    if val_ndcg > best_metric:
        best_metric = val_ndcg
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"  New best model at epoch {epoch}")
    else:
        patience_counter += 1
        print(f"  No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model from epoch {best_epoch} with Val NDCG@{EVAL_K}={best_metric:.4f}")

# ============================================================
# 10. Final test evaluation (FULL CATALOG)
# ============================================================
test_p, test_r, test_hr, test_ndcg = evaluate_full_catalog_at_k(
    model=model,
    pos_by_user=test_pos_by_user,
    seen_items_by_user=user_seen_train,
    k=EVAL_K,
    max_users=EVAL_MAX_USERS
)

print("\nFinal Test Metrics (FULL CATALOG):")
print(f"P@{EVAL_K}   = {test_p:.4f}")
print(f"R@{EVAL_K}   = {test_r:.4f}")
print(f"HR@{EVAL_K}  = {test_hr:.4f}")
print(f"NDCG@{EVAL_K}= {test_ndcg:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Train: (82782709, 12) Val: (10347839, 12) Test: (10347839, 12)
NUM_USERS (remapped): 292564
MAX_ANIME_IDX (remapped): 13008
Original master_anime shape (before remapping): (13010, 833)
Aligned master_anime shape (remapped and selected features): (13009, 833)
user_dense shape: (292564, 843)
train_pos_df shape after cap: (43569903, 12)
Included status columns: ['watch_status_Completed', 'watch_status_Currently Watching', 'watch_status_Dropped', 'watch_status_On Hold', 'watch_status_Plan to Watch']
Train positives: (43569903, 12)
Val positives: (5446278, 12)
Test positives: (5444205, 12)
Start training | users=292564 | items=13009 | POS_THRESHOLD=6.0
Epoch 01 | bpr_loss=0.0961 | Val P@10=0.1897 | Val R@10=0.1483 | Val HR@10=0.7797 | Val NDCG@10=0.2342
  New best model at epoch 1
Epoch 02 | bpr_

In [ ]:
# ============================================================
# 11. Insight / Bias / Business Analysis Block
#    Add this AFTER final full-catalog test evaluation
# ============================================================
from collections import Counter
import numpy as np
import pandas as pd
import torch

ANALYSIS_K = EVAL_K
ANALYSIS_MAX_USERS = EVAL_MAX_USERS   # set None if you want all evaluated users
PERSONALIZATION_SAMPLE_USERS = 200
LONG_TAIL_QUANTILE = 0.80   # bottom 20% by popularity = long tail

# ------------------------------------------------------------
# A. Full-catalog top-K recommendations for selected users
# ------------------------------------------------------------
@torch.no_grad()
def get_topk_recommendations_full_catalog(model, user_ids, seen_items_by_user, k=10, item_batch_size=2048):
    model.eval()

    # Your pipeline uses 1-based real item ids
    item_ids_np = np.arange(1, NUM_ITEMS, dtype=np.int64)

    all_item_vecs = []
    for start in range(0, len(item_ids_np), item_batch_size):
        batch_ids_np = item_ids_np[start:start + item_batch_size]
        batch_ids = torch.tensor(batch_ids_np, dtype=torch.long, device=DEVICE)
        batch_vecs = model.item_vector(batch_ids)
        all_item_vecs.append(batch_vecs)

    all_item_vecs = torch.cat(all_item_vecs, dim=0)  # [num_items-1, D]

    results = {}
    user_batch_size = 256

    for start in range(0, len(user_ids), user_batch_size):
        batch_users = user_ids[start:start + user_batch_size]
        u_tensor = torch.tensor(batch_users, dtype=torch.long, device=DEVICE)
        user_vecs = model.user_vector(u_tensor)  # [B, D]

        scores = torch.matmul(user_vecs, all_item_vecs.T).cpu().numpy()  # [B, num_items-1]

        for row_idx, u in enumerate(batch_users):
            u = int(u)
            seen = seen_items_by_user.get(u, set())

            row_scores = scores[row_idx].copy()

            # exclude seen training items
            if len(seen) > 0:
                seen_mask = np.isin(item_ids_np, list(seen))
                row_scores[seen_mask] = -1e12

            top_idx = np.argpartition(row_scores, -k)[-k:]
            top_idx = top_idx[np.argsort(row_scores[top_idx])[::-1]]
            top_items = item_ids_np[top_idx].tolist()

            results[u] = top_items

    return results


# ------------------------------------------------------------
# B. Compute user-level ranking metrics from recommendation dict
# ------------------------------------------------------------
def user_level_metrics_from_recs(recs_by_user, gt_by_user, k=10):
    rows = []

    for u, recs in recs_by_user.items():
        gt = gt_by_user.get(u, set())
        if len(gt) == 0:
            continue

        topk = recs[:k]
        hits = np.isin(topk, list(gt)).astype(np.float32)
        num_hits = float(hits.sum())

        precision = num_hits / k
        recall = num_hits / len(gt)
        hitrate = 1.0 if num_hits > 0 else 0.0

        denom = np.log2(np.arange(2, len(topk) + 2))
        dcg = float((hits / denom).sum())
        ideal_hits = np.ones(min(k, len(gt)), dtype=np.float32)
        idcg = float((ideal_hits / np.log2(np.arange(2, len(ideal_hits) + 2))).sum())
        ndcg = dcg / idcg if idcg > 0 else 0.0

        rows.append({
            "user_idx": u,
            "precision": precision,
            "recall": recall,
            "hitrate": hitrate,
            "ndcg": ndcg,
            "num_test_positives": len(gt)
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# C. Choose users for analysis
# ------------------------------------------------------------
analysis_users = list(test_pos_by_user.keys())
if ANALYSIS_MAX_USERS is not None and len(analysis_users) > ANALYSIS_MAX_USERS:
    np.random.seed(SEED)
    analysis_users = list(np.random.choice(analysis_users, size=ANALYSIS_MAX_USERS, replace=False))

print(f"\nRunning recommendation audit on {len(analysis_users)} users...")

audit_recs = get_topk_recommendations_full_catalog(
    model=model,
    user_ids=analysis_users,
    seen_items_by_user=user_seen_train,
    k=ANALYSIS_K
)

audit_metrics_df = user_level_metrics_from_recs(
    recs_by_user=audit_recs,
    gt_by_user=test_pos_by_user,
    k=ANALYSIS_K
)

print("\n=== User-Level Audit Metrics ===")
print(audit_metrics_df[["precision", "recall", "hitrate", "ndcg"]].mean().round(4))


# ------------------------------------------------------------
# D. Performance by User Activity Segment
# ------------------------------------------------------------
train_user_activity = train_df.groupby("user_idx").size().rename("train_interactions")
train_user_activity = train_user_activity.reindex(range(NUM_USERS), fill_value=0)

activity_df = pd.DataFrame({
    "user_idx": analysis_users,
    "train_interactions": [train_user_activity.get(u, 0) for u in analysis_users]
})

activity_df["activity_segment"] = pd.qcut(
    activity_df["train_interactions"].rank(method="first"),
    q=4,
    labels=["light", "medium", "heavy", "power"],
    duplicates="drop"
)

audit_metrics_df = audit_metrics_df.merge(activity_df, on="user_idx", how="left")

segment_perf = audit_metrics_df.groupby("activity_segment")[["precision", "recall", "hitrate", "ndcg"]].mean()
segment_counts = audit_metrics_df.groupby("activity_segment").size().rename("num_users")

print("\n=== Performance by User Activity Segment ===")
print(pd.concat([segment_perf, segment_counts], axis=1).round(4))


# ------------------------------------------------------------
# E. Popularity statistics from TRAIN
# ------------------------------------------------------------
item_pop = train_df["anime_idx"].value_counts().rename("train_popularity")
item_pop = item_pop.reindex(range(NUM_ITEMS), fill_value=0)

# keep real item ids only (1..NUM_ITEMS-1)
nonzero_item_pop = item_pop[item_pop.index != 0]
pop_threshold = nonzero_item_pop.quantile(LONG_TAIL_QUANTILE)

item_bucket = pd.Series(index=nonzero_item_pop.index, dtype="object")
item_bucket[nonzero_item_pop >= pop_threshold] = "head_mid"
item_bucket[nonzero_item_pop < pop_threshold] = "long_tail"

pop_prob = nonzero_item_pop / nonzero_item_pop.sum()
item_novelty = -np.log(pop_prob + 1e-12)


# ------------------------------------------------------------
# F. Recommendation distribution / bias metrics
# ------------------------------------------------------------
all_recommended_items = []
for u, recs in audit_recs.items():
    all_recommended_items.extend(recs[:ANALYSIS_K])

rec_counter = Counter(all_recommended_items)
unique_recommended_items = len(rec_counter)

# catalog coverage
catalog_coverage = unique_recommended_items / (NUM_ITEMS - 1)

# recommendation concentration (HHI)
rec_freq = np.array(list(rec_counter.values()), dtype=np.float64)
rec_share = rec_freq / rec_freq.sum()
recommendation_hhi = float(np.sum(rec_share ** 2))

# average popularity of recommended items
avg_rec_popularity = float(np.mean([item_pop.get(i, 0) for i in all_recommended_items]))

# long-tail share of recommendations
rec_long_tail_share = float(np.mean([
    1.0 if item_bucket.get(i, "head_mid") == "long_tail" else 0.0
    for i in all_recommended_items
]))

# novelty
avg_rec_novelty = float(np.mean([
    item_novelty.get(i, 0.0) for i in all_recommended_items
]))

# ground truth distribution for comparison
all_gt_items = []
for u in analysis_users:
    all_gt_items.extend(list(test_pos_by_user.get(u, set())))

avg_gt_popularity = float(np.mean([item_pop.get(i, 0) for i in all_gt_items])) if len(all_gt_items) > 0 else 0.0
gt_long_tail_share = float(np.mean([
    1.0 if item_bucket.get(i, "head_mid") == "long_tail" else 0.0
    for i in all_gt_items
])) if len(all_gt_items) > 0 else 0.0

popularity_gap = avg_rec_popularity - avg_gt_popularity
long_tail_gap = rec_long_tail_share - gt_long_tail_share


# ------------------------------------------------------------
# G. Personalization
# ------------------------------------------------------------
sample_users = list(audit_recs.keys())
if len(sample_users) > PERSONALIZATION_SAMPLE_USERS:
    np.random.seed(SEED)
    sample_users = list(np.random.choice(sample_users, size=PERSONALIZATION_SAMPLE_USERS, replace=False))

jaccards = []
for i in range(len(sample_users)):
    ui = sample_users[i]
    set_i = set(audit_recs[ui][:ANALYSIS_K])

    for j in range(i + 1, len(sample_users)):
        uj = sample_users[j]
        set_j = set(audit_recs[uj][:ANALYSIS_K])

        union = len(set_i | set_j)
        inter = len(set_i & set_j)
        if union > 0:
            jaccards.append(inter / union)

avg_jaccard = float(np.mean(jaccards)) if len(jaccards) > 0 else 0.0
personalization = 1.0 - avg_jaccard


# ------------------------------------------------------------
# H. Hit-rate by popularity bucket of ground-truth items
# ------------------------------------------------------------
bucket_hit_rows = []

for u, recs in audit_recs.items():
    gt = test_pos_by_user.get(u, set())
    if len(gt) == 0:
        continue

    topk = set(recs[:ANALYSIS_K])

    for item in gt:
        bucket = item_bucket.get(item, "head_mid")
        hit = 1 if item in topk else 0
        bucket_hit_rows.append({
            "user_idx": u,
            "anime_idx": item,
            "popularity_bucket": bucket,
            "hit_at_k": hit
        })

bucket_hit_df = pd.DataFrame(bucket_hit_rows)

print("\n=== Hit Rate by Ground-Truth Popularity Bucket ===")
if len(bucket_hit_df) > 0:
    pop_bucket_perf = bucket_hit_df.groupby("popularity_bucket")["hit_at_k"].mean()
    pop_bucket_counts = bucket_hit_df.groupby("popularity_bucket").size().rename("num_items")
    print(pd.concat([pop_bucket_perf.rename("hit_rate"), pop_bucket_counts], axis=1).round(4))
else:
    print("No bucket hit data available.")


# ------------------------------------------------------------
# I. Summary table
# ------------------------------------------------------------
summary_df = pd.DataFrame({
    "metric": [
        "catalog_coverage_at_k",
        "recommendation_hhi",
        "avg_recommended_item_popularity",
        "avg_ground_truth_item_popularity",
        "popularity_gap_rec_minus_truth",
        "recommended_long_tail_share",
        "ground_truth_long_tail_share",
        "long_tail_gap_rec_minus_truth",
        "avg_recommendation_novelty",
        "personalization_1_minus_avg_jaccard"
    ],
    "value": [
        catalog_coverage,
        recommendation_hhi,
        avg_rec_popularity,
        avg_gt_popularity,
        popularity_gap,
        rec_long_tail_share,
        gt_long_tail_share,
        long_tail_gap,
        avg_rec_novelty,
        personalization
    ]
})

print("\n=== Recommendation Bias / Coverage Summary ===")
print(summary_df.round(4))


Running recommendation audit on 283232 users...

=== User-Level Audit Metrics ===
precision    0.1968
recall       0.1548
hitrate      0.7917
ndcg         0.2433
dtype: float64


/tmp/ipykernel_4543/1674389136.py:148: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment_perf = audit_metrics_df.groupby("activity_segment")[["precision", "recall", "hitrate", "ndcg"]].mean()
/tmp/ipykernel_4543/1674389136.py:149: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  segment_counts = audit_metrics_df.groupby("activity_segment").size().rename("num_users")



=== Performance by User Activity Segment ===
                  precision  recall  hitrate    ndcg  num_users
activity_segment                                               
light                0.0985  0.2309   0.6003  0.1939      70808
medium               0.1711  0.1703   0.7888  0.2147      70808
heavy                0.2277  0.1310   0.8635  0.2555      70808
power                0.2898  0.0871   0.9143  0.3091      70808

=== Hit Rate by Ground-Truth Popularity Bucket ===
                   hit_rate  num_items
popularity_bucket                     
head_mid             0.1149    4737220
long_tail            0.0184     706985

=== Recommendation Bias / Coverage Summary ===
                                metric       value
0                catalog_coverage_at_k      0.4480
1                   recommendation_hhi      0.0032
2      avg_recommended_item_popularity  73229.3894
3     avg_ground_truth_item_popularity  44918.2854
4       popularity_gap_rec_minus_truth  28311.1040
5       